# Feature Engineering
This notebook creates derived features from the processed renewable energy scenario data, including coverage ratios, surplus levels, and moving averages.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

scenario_path = processed_dir / 'scenario_forecasts.csv'

print(f"Loading {scenario_path}")
scenarios_df = pd.read_csv(scenario_path)
scenarios_df['date'] = pd.to_datetime(scenarios_df['date'])
scenarios_df.head()

In [ ]:
scenarios_df['coverage_pct'] = (scenarios_df['renewable_total_mw'] / scenarios_df['load_mw'] * 100).clip(lower=0)
scenarios_df['surplus_mw'] = (scenarios_df['renewable_total_mw'] - scenarios_df['load_mw']).clip(lower=0)
scenarios_df['rolling_mean_7d'] = scenarios_df.groupby('scenario')['renewable_total_mw'].transform(lambda x: x.rolling(7, min_periods=1).mean())
scenarios_df['rolling_shortfall_7d'] = scenarios_df.groupby('scenario')['shortfall_mw'].transform(lambda x: x.rolling(7, min_periods=1).mean())

feature_path = processed_dir / 'scenario_features.csv'
scenarios_df.to_csv(feature_path, index=False)
print(f"Saved engineered features to {feature_path}")
scenarios_df[['date','scenario','coverage_pct','surplus_mw','rolling_mean_7d']].head()

### Notes
- `coverage_pct` shows how much of daily demand is met by renewable supply.
- `surplus_mw` captures the excess renewable generation available for storage.
- Rolling metrics help identify trend and seasonal behavior over a one-week window.